# Part 1

## k-means clustering: From the CIFAR-10 dataset set

In [5]:
import numpy as np
from keras.datasets import cifar10
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Step 1: Load and preprocess data
(X_train, y_train), (X_test, y_test) = cifar10.load_data() # Use Keras to load

X_train = X_train.reshape(X_train.shape[0], -1) # Reshape to 2D array
X_train = X_train.astype('float32') / 255.0 # Normalize data

# Step 2: Apply K-Means
kmeans = KMeans(n_clusters=10, random_state=0, n_init=10) # Initialize KMeans with 10 clusters
kmeans.fit(X_train)
y_pred = kmeans.predict(X_train)

# Step 3: Evaluate (using adjusted Rand index for a more appropriate measure)
score = adjusted_rand_score(y_train.flatten(), y_pred)
print(f"Adjusted Rand Index: {score}")

# Note: For accuracy, a mapping function from cluster IDs to actual labels is needed.


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
Adjusted Rand Index: 0.041737320406931434


In [13]:
import pandas as pd

# Flatten labels
true_labels = y_train.flatten()

# Create empty 10x10 table
table = np.zeros((10,10), dtype=int)

# Count occurrences
for true, cluster in zip(true_labels, y_pred):
    table[true][cluster] += 1

# Convert to DataFrame for nicer display
df = pd.DataFrame(table,
                  index=[f"class_{i}" for i in range(10)],
                  columns=[f"cluster_{i}" for i in range(10)])

print(df)

         cluster_0  cluster_1  cluster_2  cluster_3  cluster_4  cluster_5  \
class_0        131        789        973        386        222        236   
class_1        411        493        148        406        489        744   
class_2        349        179        520        310        594        918   
class_3        775        126        471        379        703        701   
class_4        949        189        326        166        602       1020   
class_5        865         90        612        271        578        565   
class_6        475         43        146        373       1080       1207   
class_7        747        209        306        558        315        654   
class_8        213       1590        514        177        194        173   
class_9        157        758        137        582        144        520   

         cluster_6  cluster_7  cluster_8  cluster_9  
class_0        487        545        941        290  
class_1        827        863        213    

## Compute Accuracy

In [14]:
# Convert to numpy if using dataframe
matrix = df.values

# For each cluster (column), get the dominant class count
correct = np.sum(np.max(matrix, axis=0))

total = np.sum(matrix)

accuracy = correct / total

print("Clustering Accuracy:", accuracy)

Clustering Accuracy: 0.22126


The clustering method achieved an accuracy of approximately 22.1%, which is significantly lower than the 84.01% test accuracy achieved in the first assignment using a supervised neural network. This difference is expected because the neural network was trained with labeled data and learned a direct mapping between image features and class labels. In contrast, K-Means clustering is an unsupervised method and groups images based only on pixel similarity without any knowledge of the true categories. Since raw pixel distances in high dimensional space do not necessarily correspond to meaningful semantic differences between objects, many clusters contain mixtures of multiple CIFAR-10 classes, resulting in substantially lower accuracy compared to the supervised approach.

# Part 2

In [16]:
# Flatten labels for easier indexing
y_train_flat = y_train.flatten()

X_subset = []
y_subset = []

for class_label in range(10):
    
    # Get indices of all images belonging to this class
    class_indices = np.where(y_train_flat == class_label)[0]
    
    # Randomly select 100 images from this class
    selected_indices = np.random.choice(class_indices, 100, replace=False)
    
    # Append to subset lists
    X_subset.append(X_train[selected_indices])
    y_subset.append(y_train_flat[selected_indices])

# Combine into single arrays
X_subset = np.concatenate(X_subset)
y_subset = np.concatenate(y_subset)

print(X_subset.shape)  # (1000, 32, 32, 3)
print(y_subset.shape)  # (1000,)\

X_subset = X_subset.reshape(X_subset.shape[0], -1)
X_subset = X_subset.astype('float32') / 255.0

kmeans = KMeans(n_clusters=10, random_state=0, n_init=10)
kmeans.fit(X_subset)

y_pred = kmeans.predict(X_subset)

(1000, 3072)
(1000,)


### Autoencoder

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential( # like the Composition layer you built
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 7)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 7),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [19]:
def train(model, num_epochs=5, batch_size=64, learning_rate=1e-3):
    torch.manual_seed(42)
    criterion = nn.MSELoss() # mean square error loss
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=learning_rate, 
                                 weight_decay=1e-5) # <--
    train_loader = torch.utils.data.DataLoader(mnist_data, 
                                               batch_size=batch_size, 
                                               shuffle=True)
    outputs = []
    for epoch in range(num_epochs):
        for data in train_loader:
            img, _ = data
            recon = model(img)
            loss = criterion(recon, img)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        print('Epoch:{}, Loss:{:.4f}'.format(epoch+1, float(loss)))
        outputs.append((epoch, img, recon),)
    return outputs